In [ ]:
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
import transformers

print(transformers.__version__)

In [ ]:
ALPHABET = ["A", "B", "C", "D", "E", "F", "G"]
choice_template = "{choice}: {state}"
prompt_template = "{question} \n \n {choices} \n \n Answer with exactly one of the letters {letters} corresponding to the correct answer."

TASKS = {
    "drawer": {
        "question": "Is the wodden drawer under the table:",
        "classes": ["open", "closed"],
        "states": ["open", "closed"],
    },
    #"light": {
    #    "question": "Which color has the rectangular led light?", 
    #    "classes": ["white", "green"],
    #    "states": ["on", "off"],
    #},

}


In [ ]:
def get_choices(states:list[str]) -> str:
    return "\n".join([f"{choice}: {state}" for choice, state in zip(ALPHABET, states)])

def get_letters(num_classes:int) -> str:
    return ", ".join(ALPHABET[:num_classes])

def get_prompt(entity:str) -> str:
    question = TASKS[entity]["question"]
    states = TASKS[entity]["states"]
    choices = get_choices(states)
    letters = get_letters(len(states))
    return prompt_template.format(question=question, choices=choices, letters=letters)

In [ ]:
prompts = []
images = []
ground_truth = []
task_names = []

# Build dataset
for entity, data in TASKS.items():
    prompt = get_prompt(entity)
    for state in data["states"]:
        for i in range(10):
            dir = "../data/samples/front"
            image = Image.open(f"{dir}/{entity}_{state}_{i}.png").convert("RGB")
            images.append(image)
            prompts.append(prompt)
            ground_truth.append(state)
            task_names.append(entity)

In [10]:
MODEL_NAME = "allenai/Molmo2-4B"

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, trust_remote_code=True, dtype=torch.float16, device_map="auto"
)

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# Batch tokenize
inputs = processor(
    images=images,
    text=prompts,
    return_tensors="pt",
    padding=True,
).to(model.device)

# Global allowed tokens
# We allow only the letters actually used
max_num_classes = max(len(task["states"]) for task in TASKS.values())
allowed_letters = ALPHABET[:max_num_classes]
allowed_token_ids = []
for label in allowed_letters:
    token_ids = processor.tokenizer.encode(label, add_special_tokens=False)
    assert len(token_ids) == 1, f"{label} is not single-token"
    allowed_token_ids.append(token_ids[0])

# Hard token constraint
def prefix_allowed_tokens_fn(batch_id, input_ids):
    return allowed_token_ids


with torch.no_grad():
    outputs = model.generate( # type: ignore
        **inputs,
        max_new_tokens=1,
        do_sample=False,
        prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
    )


# Correct extraction for padded batches
input_lengths = inputs["attention_mask"].sum(dim=1)
predicted_letters = []
for i in range(outputs.shape[0]):
    gen_tokens = outputs[i, input_lengths[i] :]
    pred = processor.tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    predicted_letters.append(pred)


# Evaluate predictions
for i in range(len(predicted_letters)):
    entity = task_names[i]
    states = TASKS[entity]["states"]
    letter_to_state = {ALPHABET[j]: state for j, state in enumerate(states)}
    predicted_state = letter_to_state.get(predicted_letters[i], "INVALID")

    print(
        f"TASK={entity:<10} "
        f"GT={ground_truth[i]:<10} "
        f"PRED={predicted_state:<10} "
        f"LETTER={predicted_letters[i]}"
    )